In [1]:
import os, sys, shutil, platform
from pathlib import Path

print('Machine:', platform.node(), '|', platform.platform())
import torch
print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

INP = Path('/kaggle/input')

# Discover the code dataset (a dir containing both src/ and configs/) — the
# mount path is nested and non-obvious, so search rather than hardcode.
code = None
for p in INP.rglob('configs'):
    if p.is_dir() and (p.parent / 'src').is_dir():
        code = p.parent
        break
assert code, 'code dataset (src/ + configs/) not found under /kaggle/input'
print('code root:', code)

# Discover the UAVVaste data: annotations.json with a sibling images/ dir.
ann = imgs = None
for a in INP.rglob('annotations.json'):
    if (a.parent.parent / 'images').is_dir():
        ann, imgs = a, a.parent.parent / 'images'
        break
assert ann and imgs, 'data (images/ + annotations/annotations.json) not found'
print('data images:', imgs, '| ann:', ann)

# Copy code into a writable working dir
work = Path('/kaggle/working/repo')
if work.exists():
    shutil.rmtree(work)
work.mkdir(parents=True)
for item in code.iterdir():
    dst = work / item.name
    shutil.copytree(item, dst) if item.is_dir() else shutil.copy2(item, dst)
os.chdir(work)
sys.path.insert(0, str(work))
print('code copied:', sorted(p.name for p in work.iterdir()))

# Link images (read-only) and copy annotations into ./data
Path('data/annotations').mkdir(parents=True, exist_ok=True)
if not Path('data/images').exists():
    os.symlink(imgs, 'data/images')
shutil.copy2(ann, 'data/annotations/annotations.json')
print('linked', len(list(Path('data/images').glob('*.*'))), 'images')


Machine: cae23670673e | Linux-6.12.90+-x86_64-with-glibc2.35
PyTorch: 2.10.0+cu128 | CUDA: True
GPU: Tesla T4
code root: /kaggle/input/datasets/sajanmahat/stw7088-code
data images: /kaggle/input/datasets/sajanmahat/litter-data/data/images | ann: /kaggle/input/datasets/sajanmahat/litter-data/data/annotations/annotations.json
code copied: ['README.md', 'configs', 'requirements.txt', 'results', 'src', 'tests']
linked 773 images


In [2]:
!pip install -q ultralytics sahi pycocotools

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.6/148.6 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 7.4 MB/s eta 0:00:00


In [3]:
!python -m src.data.splits

=== Image-level splits ===
  train: 540 images (69.9%)
  val: 116 images (15.0%)
  test: 116 images (15.0%)
  disjoint: OK
[done] written to /kaggle/working/repo/data/processed/splits.json


In [4]:
!python -m src.data.coco_to_yolo

=== COCO -> YOLO conversion ===
train: 100%|██████████████████████████████████| 540/540 [00:37<00:00, 14.54it/s]
  train: 540 images, 2608 boxes
  val: 100%|██████████████████████████████████| 116/116 [00:06<00:00, 16.80it/s]
  val: 116 images, 507 boxes
 test: 100%|██████████████████████████████████| 116/116 [00:08<00:00, 14.17it/s]
  test: 116 images, 603 boxes
[done] data.yaml -> /kaggle/working/repo/data/yolo/data.yaml
[done] YOLO dataset ready


In [5]:
!python -m src.training.train_yolo --config configs/yolo_res1024.yaml

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
=== YOLO training ===
  model: yolov8s.pt | imgsz: 1024 | epochs: 80
Ultralytics 8.4.103 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/repo/data/yolo/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=80, erasing=0.4, exist_ok=True,

In [6]:
!python -m src.training.train_yolo --config configs/yolo_optimized.yaml

=== YOLO training ===
  model: yolov8s.pt | imgsz: 1024 | epochs: 80
Ultralytics 8.4.103 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/repo/data/yolo/data.yaml, degrees=10.0, deterministic=True, device=, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=80, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.5, imgsz=1024, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo

In [7]:
!python -m src.evaluation.evaluate_yolo --config configs/yolo_res1024.yaml --tag +resolution

=== YOLO evaluation ===
  checkpoint: /kaggle/working/repo/checkpoints/yolo_res1024.pt | split: test | imgsz: 1024
Ultralytics 8.4.103 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
Model summary (fused): 73 layers, 11,125,971 parameters, 0 gradients, 28.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2666.0±1726.7 MB/s, size: 3184.1 KB)
val: Scanning /kaggle/working/repo/data/yolo/labels/test... 116 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 116/116 1.2Kit/s 0.1s
val: New cache created: /kaggle/working/repo/data/yolo/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 1.5it/s 5.5s
                   all        116        603      0.857      0.801      0.858      0.504
Speed: 4.1ms preprocess, 13.4ms inference, 0.0ms loss, 6.0ms postprocess per image
Results saved to /kaggle/working/repo/runs/detect/runs/detect/yolo_res1024_eval_test
[table] /kaggle/working/repo/results/ta

In [8]:
!python -m src.evaluation.evaluate_yolo --config configs/yolo_optimized.yaml --tag +augmentation

=== YOLO evaluation ===
  checkpoint: /kaggle/working/repo/checkpoints/yolo_optimized.pt | split: test | imgsz: 1024
Ultralytics 8.4.103 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
Model summary (fused): 73 layers, 11,125,971 parameters, 0 gradients, 28.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2791.6±2280.4 MB/s, size: 3184.1 KB)
val: Scanning /kaggle/working/repo/data/yolo/labels/test.cache... 116 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 116/116 18.0Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 1.4it/s 5.7s
                   all        116        603      0.879      0.779      0.863      0.505
Speed: 3.2ms preprocess, 14.8ms inference, 0.0ms loss, 4.5ms postprocess per image
Results saved to /kaggle/working/repo/runs/detect/runs/detect/yolo_optimized_eval_test
[table] /kaggle/working/repo/results/tables/yolo_optimized_metrics.csv
  config: +augmentation
  mode

In [9]:
!python -m src.evaluation.sahi_eval --config configs/yolo_optimized.yaml

=== SAHI evaluation ===
  checkpoint: /kaggle/working/repo/checkpoints/yolo_optimized.pt | sliced: True | images: 116 | device: cuda:0
predict: 100%|████████████████████████████████| 116/116 [03:24<00:00,  1.76s/it]
[table] /kaggle/working/repo/results/tables/yolo_sahi_metrics.csv
  config: +SAHI
  model: yolov8s.pt
  imgsz: 1024
  split: test
  sahi: True
  mAP50: 0.6229
  mAP50_95: 0.3218
  precision: 0.1164
  recall: 0.8574


In [10]:
!python -m src.evaluation.sahi_eval --config configs/yolo_optimized.yaml --no-slice

=== SAHI evaluation ===
  checkpoint: /kaggle/working/repo/checkpoints/yolo_optimized.pt | sliced: False | images: 116 | device: cuda:0
predict: 100%|████████████████████████████████| 116/116 [00:28<00:00,  4.03it/s]
[table] /kaggle/working/repo/results/tables/yolo_sahi_metrics_noslice.csv
  config: +augmentation (coco-eval)
  model: yolov8s.pt
  imgsz: 1024
  split: test
  sahi: False
  mAP50: 0.8205
  mAP50_95: 0.4365
  precision: 0.6931
  recall: 0.869


In [11]:
!python -m src.evaluation.ablation

=== Task 2 ablation ===
               arm  imgsz  mAP50  mAP50_95  precision  recall  d_mAP50  d_mAP50_95  d_precision  d_recall
    baseline (640)    640 0.7931    0.4465     0.8123  0.7536   0.0000      0.0000       0.0000    0.0000
+resolution (1024)   1024 0.8575    0.5038     0.8566  0.8010   0.0644      0.0573       0.0443    0.0474
     +augmentation   1024 0.8633    0.5046     0.8789  0.7794   0.0702      0.0581       0.0666    0.0258
             +SAHI   1024 0.6229    0.3218     0.1164  0.8574  -0.1702     -0.1247      -0.6959    0.1038
[table] /kaggle/working/repo/results/tables/yolo_ablation.csv
[fig] saved /kaggle/working/repo/results/figures/yolo_ablation.png
[done] ablation consolidated


In [12]:
import shutil
from pathlib import Path
work = Path('/kaggle/working/repo')
out = Path('/kaggle/working')
# Promote only the artefacts we want downloaded, then drop the heavy repo dir
# (tile crops, code) so `kaggle kernels output` stays small.
for sub in ['results', 'checkpoints']:
    src = work / sub
    if src.exists():
        dst = out / sub
        if dst.exists():
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
        for f in sorted(dst.rglob('*')):
            if f.is_file():
                print(f.relative_to(out), f'({f.stat().st_size//1024} KB)')
import os
os.chdir(out)
shutil.rmtree(work, ignore_errors=True)
print('done; kept:', sorted(p.name for p in out.iterdir()))


results/figures/yolo_ablation.png (81 KB)
results/figures/yolo_optimized_confusion_matrix.png (103 KB)
results/figures/yolo_optimized_confusion_matrix_normalized.png (96 KB)
results/figures/yolo_optimized_f1_curve.png (107 KB)
results/figures/yolo_optimized_labels.jpg (128 KB)
results/figures/yolo_optimized_pr_curve.png (87 KB)
results/figures/yolo_optimized_results.png (265 KB)
results/figures/yolo_res1024_confusion_matrix.png (95 KB)
results/figures/yolo_res1024_confusion_matrix_normalized.png (96 KB)
results/figures/yolo_res1024_f1_curve.png (109 KB)
results/figures/yolo_res1024_labels.jpg (128 KB)
results/figures/yolo_res1024_pr_curve.png (86 KB)
results/figures/yolo_res1024_results.png (312 KB)
results/tables/yolo_ablation.csv (0 KB)
results/tables/yolo_baseline_metrics.csv (0 KB)
results/tables/yolo_baseline_val_metrics.csv (0 KB)
results/tables/yolo_optimized_metrics.csv (0 KB)
results/tables/yolo_res1024_metrics.csv (0 KB)
results/tables/yolo_sahi_metrics.csv (0 KB)
results/tab